# 03b · Temporal action models (Step 5, TensorFlow / Keras)

**Owner:** Ben · **Input:** the clips from `03a` and the fine-tuned baseline from `03_actions` (Step 4) · **Output:** sequence models that read **all 16 frames** of a player clip, their test scores next to the single-frame baseline, and the best model saved for the pipeline (Step 8).

**Why look at more than one frame?** A single frame often can't tell a spike from a block: both are a player in the air at the net. The difference is in the movement: which way the arms swing, whether the player jumps towards the ball or into it. Sequence models see that.

### The plan: extract features once, then train sequence models

Running a CNN on 16 frames of every clip in every epoch would cost 16× the baseline's training time. Instead:

1. **Extract features once (5.3).** Pass every frame through the backbone from Step 4 (frozen) and save its 1,280-number `features` vector. Each clip becomes a 16 × 1,280 array. This is the book's transfer learning: a pretrained CNN used as a fixed feature extractor.
2. **Train sequence models on those arrays (5.4–5.5).** Each epoch now takes seconds, so we can compare several of the book's sequence models fairly.

### How this follows the course textbook (*The Deep Learning Workshop*)

| Section | Technique | Textbook |
|---|---|---|
| 5.3 | Pretrained CNN as a frozen feature extractor | Ch. 3, *Transfer Learning* |
| 5.4 | `Conv1D` + `MaxPooling1D` + `Flatten` over a sequence | Ch. 5, *1D Convolutions for Sequence Processing*, Ex. 5.04 |
| 5.4 | Hybrid: `Conv1D` followed by a recurrent layer | Ch. 5, *Using 1D Convnets with RNNs*, Ex. 5.05 |
| 5.4 | `LSTM`, `GRU`, `Bidirectional(LSTM)` | Ch. 6, *LSTMs, GRUs, and Advanced RNNs*, Ex. 6.02–6.04 |
| 5.4 | `Dropout`, `Dense` softmax, `Adam`, `sparse_categorical_crossentropy` | Ch. 2–3 |
| 5.7 | `model.save` / `load_model` | Ch. 3, *Saving and Restoring Models* |

**Beyond the book**, explained where used: a "frame average" control model (`GlobalAveragePooling1D`, no notion of order) to show what temporal order adds; mirrored training clips as augmentation (features can't be augmented on the fly); `class_weight`; macro-F1; choosing models by validation, never test.

Runtime: **GPU** for 5.3 (feature extraction); the rest runs fine on either.

## 5.1 Setup

In [ ]:
import os, sys, json, time, random, shutil
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import applications as apps
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
SEED = 0
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU") or "none")

In [ ]:
env = os.environ.get
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/volleyball/action")
    CLIPS_DIR = Path("/content/action_clips")
    if not (CLIPS_DIR / "labels.csv").exists():
        t0 = time.time(); shutil.copytree(DRIVE / "data" / "action_clips", CLIPS_DIR)
        print(f"Copied clips to local disk in {time.time() - t0:.0f} s")
    OUT = DRIVE
elif IN_KAGGLE:
    CLIPS_DIR = Path(env("CLIPS_DIR", "/kaggle/input/action-clips/action_clips"))
    OUT = Path("/kaggle/working")
else:
    CLIPS_DIR = Path(env("CLIPS_DIR", "./work/action_clips"))
    OUT = Path(env("OUT_DIR", "./action_out"))

BACKBONE = env("BACKBONE", "efficientnetb0")       # must match the Step 4 run you want to build on
BASELINE_MODEL = Path(env("BASELINE_MODEL", OUT / "checkpoints" / f"baseline_{BACKBONE}" / f"baseline_{BACKBONE}.keras"))
FEAT_DIR = OUT / "features" / BACKBONE
TEMP_DIR = OUT / "checkpoints" / "temporal"
RESULTS_DIR = OUT / "results" / "temporal"
for d in (FEAT_DIR, TEMP_DIR, RESULTS_DIR): d.mkdir(parents=True, exist_ok=True)

HP = dict(
    backbone=BACKBONE,
    use_finetuned_backbone=True,    # True: Step 4's fine-tuned backbone; False: plain ImageNet weights
    flip_augment=True,              # also extract features of mirrored training clips (doubles the training set)
    frame_sets={"16 frames": list(range(16)),          # frames -8 .. +7 around the labelled frame
                "10 frames": list(range(3, 13))},      # frames -5 .. +4, the window the dataset's authors used
    models=env("MODELS", "frame_average,conv1d,lstm,gru,conv1d_lstm,bilstm").split(","),
    units=128, input_dropout=0.3, dropout=0.5,
    lr=1e-3, batch_size=64,
    max_epochs=int(env("EPOCHS", 60)), patience=int(env("PATIENCE", 8)),
)
CLASSES = ["spike", "set", "dig", "block", "other"]
PREPROCESS = {"efficientnetb0": apps.efficientnet.preprocess_input, "vgg16": apps.vgg16.preprocess_input,
              "resnet50": apps.resnet50.preprocess_input}[BACKBONE]
json.dump(HP, open(RESULTS_DIR / "hparams.json", "w"), indent=2)
print("baseline model:", BASELINE_MODEL, "(found)" if BASELINE_MODEL.exists() else "(NOT FOUND)")
print("features ->", FEAT_DIR); print(HP)

## 5.2 The feature extractor

We reuse the backbone **fine-tuned in Step 4**: its last block has already adapted to volleyball players, so its features suit our clips better than plain ImageNet ones. Step 4 saved the whole model; we load it (Ch. 3, *Saving and Restoring Models*) and cut it off at the layer named `features`, the 1,280-number vector just before the classification head.

If you set `use_finetuned_backbone = False`, or the Step 4 file isn't there, the plain ImageNet backbone is used instead. That makes a fair comparison row if you want one, but it's usually a little worse.

In [ ]:
if HP["use_finetuned_backbone"] and BASELINE_MODEL.exists():
    baseline = tf.keras.models.load_model(BASELINE_MODEL)
    extractor = tf.keras.Model(baseline.input, baseline.get_layer("features").output, name="feature_extractor")
    source = f"fine-tuned in Step 4 ({BASELINE_MODEL.name})"
else:
    cls = {"efficientnetb0": apps.EfficientNetB0, "vgg16": apps.VGG16, "resnet50": apps.ResNet50}[BACKBONE]
    base = cls(weights="imagenet", include_top=False, pooling="avg", input_shape=(112, 112, 3))
    inp = layers.Input((112, 112, 3)); extractor = tf.keras.Model(inp, base(inp, training=False), name="feature_extractor")
    source = "ImageNet only (Step 4 model not used)"
FEAT_DIM = extractor.output_shape[-1]
extractor.save(TEMP_DIR / f"feature_extractor_{BACKBONE}.keras")      # Step 8 needs it
print(f"Feature extractor: {BACKBONE}, {source}; {FEAT_DIM} numbers per frame")

## 5.3 Extract features once

For every clip: 16 frames → preprocessing → extractor → a 16 × 1,280 array, stored as `float16` to halve the size. For the training split we also extract features of the **horizontally mirrored** clip, the same flip augmentation as Step 4, done once here because features can't be augmented on the fly.

This is the only slow cell (one CNN pass per frame; a GPU helps a lot). It saves one file per split and skips splits already done, so a disconnect only costs the split in progress.

In [ ]:
labels = pd.read_csv(CLIPS_DIR / "labels.csv")
class_weights = json.load(open(CLIPS_DIR / "class_weights.json"))

def clip_features(clips):                      # clips: (n, 16, 112, 112, 3) uint8
    n, T = clips.shape[:2]
    frames = PREPROCESS(clips.reshape(n * T, *clips.shape[2:]).astype("float32"))
    return extractor.predict(frames, batch_size=256, verbose=0).reshape(n, T, -1).astype("float16")

def extract_split(split, chunk=32):
    path = FEAT_DIR / f"{split}.npz"
    if path.exists():
        print(f"{split}: already extracted"); return
    g = labels[labels.split == split].sort_values(["shard", "idx"])
    F, F_flip, y = [], [], []
    t0 = time.time()
    for shard, rows in tqdm(list(g.groupby("shard")), desc=split):
        z = np.load(CLIPS_DIR / f"{split}_{shard:03d}.npz")
        X, Y = z["X"][rows.idx.values], z["y"][rows.idx.values]
        for i in range(0, len(X), chunk):
            F.append(clip_features(X[i:i + chunk]))
            if split == "train" and HP["flip_augment"]:
                F_flip.append(clip_features(X[i:i + chunk][:, :, :, ::-1]))   # mirror left-right
        y.append(Y)
    out = dict(F=np.concatenate(F), y=np.concatenate(y).astype("int64"))
    if F_flip: out["F_flip"] = np.concatenate(F_flip)
    np.savez(path, **out)
    print(f"{split}: {out['F'].shape} in {time.time() - t0:.0f} s")

for s in ["train", "val", "test"]:
    extract_split(s)

feats = {s: dict(np.load(FEAT_DIR / f"{s}.npz")) for s in ["train", "val", "test"]}
for s, d in feats.items():
    print(f"{s:5s} F={d['F'].shape} counts={dict(zip(CLASSES, np.bincount(d['y'], minlength=5).tolist()))}"
          + (f" (+{len(d['F_flip'])} mirrored)" if "F_flip" in d else ""))
print("feature spread (should be clearly above 0):", float(feats["train"]["F"].astype("float32").std(axis=0).mean()))

**Check:** counts match `03a`; the train split shows the mirrored copies; the feature spread is clearly above zero (if it's ~0, every clip looks the same to the extractor and something is wrong with the weights or preprocessing).

## 5.4 The sequence models

Every model takes a clip as a sequence of frame features, shape `(frames, 1280)`, and ends in `Dense(5, softmax)`. A `Dropout(0.3)` on the input and `Dropout(0.5)` before the output (Ch. 2) limit overfitting: our clips are far fewer than the 1,280 numbers per frame.

| Name | Layers | Textbook | What it tests |
|---|---|---|---|
| `frame_average` | average over frames → Dense | (control, not in the book) | How far you get with **no** notion of order |
| `conv1d` | Conv1D(128, 3) → MaxPooling1D(2) → Conv1D(128, 3) → Flatten → Dense | Ch. 5, Ex. 5.04 | Short local motion patterns (a few frames) |
| `lstm` | LSTM(128) → Dense | Ch. 6, Ex. 6.02 | Order over the whole clip |
| `gru` | GRU(128) → Dense | Ch. 6, Ex. 6.03 | A lighter recurrent layer |
| `conv1d_lstm` | Conv1D(128, 3) → LSTM(128) → Dense | Ch. 5, Ex. 5.05 (hybrid) | Local patterns, then their order |
| `bilstm` | Bidirectional(LSTM(64)) → Dense | Ch. 6, Ex. 6.04 | Reading the clip forwards and backwards |

In [ ]:
def build(name, T, D=None):
    D = D or FEAT_DIM; u = HP["units"]
    inp = layers.Input((T, D))
    x = layers.Dropout(HP["input_dropout"])(inp)
    if name == "frame_average":
        x = layers.GlobalAveragePooling1D()(x)
    elif name == "conv1d":
        x = layers.Conv1D(u, 3, activation="relu")(x)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(u, 3, activation="relu")(x)
        x = layers.Flatten()(x)
    elif name == "lstm":
        x = layers.LSTM(u)(x)
    elif name == "gru":
        x = layers.GRU(u)(x)
    elif name == "conv1d_lstm":
        x = layers.Conv1D(u, 3, activation="relu")(x)
        x = layers.LSTM(u)(x)
    elif name == "bilstm":
        x = layers.Bidirectional(layers.LSTM(u // 2))(x)
    else:
        raise ValueError(name)
    x = layers.Dropout(HP["dropout"])(x)
    out = layers.Dense(len(CLASSES), activation="softmax")(x)
    m = tf.keras.Model(inp, out, name=name)
    m.compile(loss="sparse_categorical_crossentropy", optimizer=tf.keras.optimizers.Adam(HP["lr"]), metrics=["accuracy"])
    return m

for n in HP["models"]:
    print(f"{n:14s} {build(n, 16).count_params():>9,} parameters")

## 5.5 Train and compare

Each model is trained on both frame windows (16 and 10 frames). Training is the same loop as Step 4, now taking seconds per epoch: one epoch, then **validation macro-F1**; the best weights are kept; training stops after `patience` epochs without improvement. Then the best weights are scored **once** on the test split.

Results go to `temporal_results.csv` as each run finishes, and finished runs are skipped if you re-run the cell.

In [ ]:
CW = {i: class_weights[c] for i, c in enumerate(CLASSES)}
RES = RESULTS_DIR / "temporal_results.csv"
results = pd.read_csv(RES) if RES.exists() else pd.DataFrame()

def split_xy(split, idx):
    d = feats[split]
    X, y = d["F"][:, idx].astype("float32"), d["y"]
    if split == "train" and "F_flip" in d:
        X, y = np.concatenate([X, d["F_flip"][:, idx].astype("float32")]), np.concatenate([y, y])
    return X, y

def macro_f1(y, p): return f1_score(y, p, average="macro", labels=range(len(CLASSES)), zero_division=0)

def train_one(name, frames_label):
    idx = HP["frame_sets"][frames_label]
    Xtr, ytr = split_xy("train", idx); Xva, yva = split_xy("val", idx); Xte, yte = split_xy("test", idx)
    tf.random.set_seed(SEED)
    m = build(name, len(idx))
    best_f1, best_w, best_ep, stale, curve, t0 = -1, None, -1, 0, [], time.time()
    for ep in range(HP["max_epochs"]):
        h = m.fit(Xtr, ytr, batch_size=HP["batch_size"], epochs=1, class_weight=CW, shuffle=True, verbose=0)
        f1 = macro_f1(yva, m.predict(Xva, verbose=0).argmax(1))
        curve.append((h.history["loss"][0], f1))
        if f1 > best_f1: best_f1, best_w, best_ep, stale = f1, m.get_weights(), ep, 0
        else: stale += 1
        if stale >= HP["patience"]: break
    m.set_weights(best_w)
    pte = m.predict(Xte, verbose=0).argmax(1)
    rep = classification_report(yte, pte, labels=range(len(CLASSES)), target_names=CLASSES, output_dict=True, zero_division=0)
    m.save(TEMP_DIR / f"{name}_{len(idx)}f.keras")
    return dict(model=name, frames=frames_label, val_macro_f1=round(best_f1, 4), test_macro_f1=round(macro_f1(yte, pte), 4),
                test_acc=round(float((pte == yte).mean()), 4), best_epoch=best_ep, epochs_run=len(curve),
                secs=round(time.time() - t0, 1), **{f"{c}_f1": round(rep[c]["f1-score"], 4) for c in CLASSES}), curve

curves = {}
for frames_label in HP["frame_sets"]:
    for name in HP["models"]:
        if len(results) and ((results.model == name) & (results.frames == frames_label)).any():
            print(f"skip {name} / {frames_label} (done)"); continue
        row, curves[(name, frames_label)] = train_one(name, frames_label)
        results = pd.concat([results, pd.DataFrame([row])], ignore_index=True); results.to_csv(RES, index=False)
        print(f"{name:14s} {frames_label}: val macro-F1 {row['val_macro_f1']:.3f} | test macro-F1 {row['test_macro_f1']:.3f} "
              f"| best epoch {row['best_epoch']} of {row['epochs_run']} | {row['secs']:.0f}s")
results.sort_values("val_macro_f1", ascending=False)

## 5.6 Compare with the single-frame baseline

The chart puts every model next to the Step 4 baseline (dashed line). What to look for:

- **`frame_average` above the baseline:** seeing several frames helps even without order (more views of the same moment).
- **`lstm`, `gru`, `conv1d_lstm` or `conv1d` above `frame_average`:** the **order** of frames carries information, i.e. the model is using the motion. This is the key result for your report.
- **16 vs 10 frames:** whether the wider window adds anything. The dataset's authors used 10.
- **Per-class F1:** spike and block are where temporal models should gain most.

In [ ]:
summary_path = OUT / "results" / "summary.csv"
base_rows = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
base_f1 = float(base_rows[base_rows["input"] == "1 frame"]["test_macro_f1"].max()) if len(base_rows) else np.nan

r = results.copy()
order = [m for m in HP["models"] if m in set(r.model)]
fig, ax = plt.subplots(figsize=(11, 4.5))
w = 0.38
for k, (fl, off) in enumerate(zip(HP["frame_sets"], (-w / 2, w / 2))):
    vals = [r[(r.model == m) & (r.frames == fl)]["test_macro_f1"].max() for m in order]
    bars = ax.bar(np.arange(len(order)) + off, vals, w, label=fl)
    ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
if not np.isnan(base_f1):
    ax.axhline(base_f1, ls="--", c="gray", label=f"single-frame baseline ({base_f1:.2f})")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order); ax.set_ylim(0, 1); ax.set_ylabel("test macro-F1")
ax.set_title(f"Temporal models on {BACKBONE} features"); ax.legend(loc="upper left")
fig.tight_layout(); fig.savefig(RESULTS_DIR / "model_comparison.png", dpi=100); plt.show()

per_class = r.set_index(["model", "frames"])[[f"{c}_f1" for c in CLASSES]].round(2)
per_class

## 5.7 The best model: final scores and saving for the pipeline

The best model is chosen by **validation** macro-F1. The test split only reports its score; picking by test would inflate the result you report. We then:

1. show its test confusion matrix and per-class scores;
2. write its rows into the shared `summary.csv` (the Step 6 table), next to the baseline;
3. save what Step 8 needs: the sequence model, the feature extractor (saved in 5.2) and a small info file;
4. check that `classify_clips()`, the function Step 8 will use on our own videos, gives the same answers on raw test clips as the stored features did.

In [ ]:
best = results.sort_values("val_macro_f1", ascending=False).iloc[0]
idx = HP["frame_sets"][best.frames]
best_model = tf.keras.models.load_model(TEMP_DIR / f"{best.model}_{len(idx)}f.keras")
Xte, yte = split_xy("test", idx)
pte = best_model.predict(Xte, verbose=0).argmax(1)
print(f"Best by validation: {best.model}, {best.frames} (val macro-F1 {best.val_macro_f1:.3f})")
print(f"TEST macro-F1 {macro_f1(yte, pte):.3f} | accuracy {(pte == yte).mean():.3f}\n")
print(classification_report(yte, pte, labels=range(len(CLASSES)), target_names=CLASSES, digits=3, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(yte, pte, labels=range(len(CLASSES)))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=axes[0], colorbar=False, cmap="Blues"); axes[0].set_title("Counts")
ConfusionMatrixDisplay((cm / cm.sum(1, keepdims=True).clip(min=1)).round(2), display_labels=CLASSES).plot(ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title("Row-normalised (recall)")
for ax in axes: ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.suptitle(f"{BACKBONE} + {best.model} ({best.frames}) · test macro-F1 {macro_f1(yte, pte):.3f}"); fig.tight_layout()
fig.savefig(RESULTS_DIR / "best_confusion_matrix.png", dpi=100); plt.show()

# Step 6 table rows
rows = [{"model": f"{BACKBONE} + {x['model']}", "input": x["frames"], "run": f"temporal_{x['model']}_{x['frames'].split()[0]}f",
         "test_macro_f1": x["test_macro_f1"], "test_acc": x["test_acc"], **{f"{c}_f1": x[f"{c}_f1"] for c in CLASSES}}
        for x in results.to_dict("records")]
summary = pd.concat([base_rows[~base_rows["run"].astype(str).str.startswith("temporal_")] if len(base_rows) else base_rows,
                     pd.DataFrame(rows)], ignore_index=True)
summary.to_csv(summary_path, index=False)

# What Step 8 needs
best_model.save(TEMP_DIR / "best_action_model.keras")
info = dict(backbone=BACKBONE, sequence_model=best.model, frames=best.frames, frame_indices=idx, clip_frames=16,
            crop_size=112, classes=CLASSES, feature_extractor=f"feature_extractor_{BACKBONE}.keras",
            sequence_model_file="best_action_model.keras", val_macro_f1=float(best.val_macro_f1), test_macro_f1=float(best.test_macro_f1))
json.dump(info, open(TEMP_DIR / "action_model_info.json", "w"), indent=2)
print("saved:", [p.name for p in TEMP_DIR.glob("*") if p.name.startswith(("best_", "feature_", "action_"))])

In [ ]:
def classify_clips(clips, extractor=extractor, seq_model=best_model, info=info):
    # clips: (n, 16, 112, 112, 3) uint8 RGB crops made with 03a's square_crop. Returns (labels, confidences).
    F = clip_features(clips)[:, info["frame_indices"]].astype("float32")
    p = seq_model.predict(F, verbose=0)
    return [info["classes"][k] for k in p.argmax(1)], p.max(1)

# Check on a few raw test clips: same answers as the stored features gave
g = labels[labels.split == "test"].sort_values(["shard", "idx"])
z = np.load(CLIPS_DIR / f"test_{int(g.shard.iloc[0]):03d}.npz")
n = min(8, len(z["X"]))
pred, conf = classify_clips(z["X"][:n])
print("raw clips :", pred); print("features  :", [CLASSES[k] for k in pte[:n]]); print("confidence:", np.round(conf, 2))

**Check:** the two rows of labels match. Small confidence differences are fine (features were stored as `float16`).

## Optional: end-to-end fine-tuning

The features above come from a backbone that was fine-tuned on **single frames** and then frozen. Training the backbone and the sequence model together could add a little more, but it means running the CNN on all 16 frames of every clip in every epoch, many times slower. Only consider it if you have time left after Steps 7–9, and only for the best model here.

## Done checklist (Step 5)

- [ ] 5.3: features extracted for all splits; the spread check is clearly above 0
- [ ] 5.5: all models trained on both frame windows; `temporal_results.csv` saved
- [ ] 5.6: `model_comparison.png` and the per-class table saved for the report
- [ ] 5.7: best model chosen by validation; its confusion matrix saved; `summary.csv` now holds the Step 6 table
- [ ] 5.7: `best_action_model.keras`, `feature_extractor_<backbone>.keras` and `action_model_info.json` on Drive (Step 8 uses them); the raw-clip check matches
- [ ] Commit this notebook and the `results/temporal/` CSV and PNG files; keep `.keras` and `.npz` files on Drive only

**For your report:** the comparison chart and the per-class table are your main results. The story to tell is *baseline → frame average → best temporal model*: how much more frames help, how much **order** helps, and which actions gain most.